# 08 · SLE (GSE65391) compared with JDM and the interferonopathies, each against its own healthy controls

**Run order:** after 001 to 003 (public array data) and 05 and 07 (your differential expression and JDM results). Kernel: **R (endotypes)**.

## The question
How does the blood signature of childhood SLE compare with CANDLE, SAVI, AGS, UAID and JDM?

## Why it is done this way (Anne's lesson, notebooks 46, 48, 49)
SLE here is a **microarray** (GSE65391, Illumina HT-12, whole blood); your diseases are **RNA-seq**. The numbers sit on different scales, so SLE is **not** pooled with them and no lab correction is attempted: Anne tried to remove the lab difference between SLE and the internal data (RUVg, notebook 49) and found that no setting separated the lab from the disease. What can be compared fairly is **each disease against the healthy controls of its own dataset**:
- SLE against the 46 healthy children **of the same array study**;
- CANDLE, SAVI, AGS, UAID against your healthy controls (notebook 05);
- JDM against the JDM lab's healthy controls (notebook 07).

Each gives a list of genes up or down *relative to healthy, in the same lab*. Those lists are put side by side by gene symbol: rank correlation of fold changes, shared top genes, direction, and the interferon score. Fold changes are compared by **rank and direction, not size**, because array and RNA-seq compress fold changes differently.

## What this notebook does, in order
1. Reads your `probes.rds` and `metadata.rds` (001 to 003).
2. **Bridges the two array batches** with the 23 technical-replicate pairs, as Anne's notebook 03 does.
3. **Probes to genes**: Perfect/Good probes with an Entrez ID; per gene, the probe with the highest mean (MaxMean), as Anne's 03b does. No WGCNA package is needed.
4. **SLE vs healthy** (limma): first visit of each child with SLE, earliest visit of each healthy child, as Anne's 48.
5. Compares with every disease of 05 and with JDM, with and without the 28 interferon genes.
6. Interferon score of SLE and its healthy children.

**Reads:** `data/run_artifacts/GSE65391/probes.rds` and `metadata.rds`; `data/GSE65391/GPL10558.soft.gz`; `data/ncbi/Homo_sapiens.gene_info.gz`; `rnaseq_out_v2/differential_expression/de_*_vs_HC.csv` (05); `rnaseq_out_v2/jdm_comparison/de_JDM_vs_HC_jdm_lab.csv` (07).
**Writes (all CSV):** `rnaseq_out_v2/sle_comparison/`: `de_SLE_vs_healthy_GSE65391.csv`, `comparison_sle_vs_others.csv`, `ifn_score_sle.csv`, `PROCESSING_LOG_sle.md`.

---
# Part 1 · Settings
| variable | default | meaning |
|---|---|---|
| `ALPHA` | `0.05` | adjusted p-value cut-off for "top genes" |
| `LFC` | `1` | \|log2 fold change\| cut-off for your RNA-seq diseases (as in 05 and 07) |
| `JDM_LFC` | `0.5` | cut-off for JDM (as in 07) |
| `SLE_LFC` | `0.5` | cut-off for SLE; array fold changes are compressed, so a lower cut-off than RNA-seq |
| `SKIP_DISEASES` | `AGS-like`, `CANDLE-like` | groups of 2 to 3 samples, left out of the comparison; set to `character(0)` to include them |

In [ ]:
RESULTS <- "/home/jovyan/session_data/Attempt0/endotype_results"
IN_DIR  <- file.path(RESULTS, "rnaseq_out_v2")
DE_DIR  <- file.path(IN_DIR, "differential_expression")        # from notebook 05
JDM_DE  <- file.path(IN_DIR, "jdm_comparison", "de_JDM_vs_HC_jdm_lab.csv")   # from notebook 07
OUT_DIR <- file.path(IN_DIR, "sle_comparison")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)
GENE_SET_DIR <- "/home/jovyan/session_data/endotype-transcripts-007/genes"

ALPHA <- 0.05; LFC <- 1; JDM_LFC <- 0.5; SLE_LFC <- 0.5
SKIP_DISEASES <- c("AGS-like", "CANDLE-like")

source("/home/jovyan/session_data/Attempt0/src/paths.R")        # gives art(), raw(), NCBI, SEED
suppressMessages({library(limma); library(GEOquery)})

need_files <- c(art("GSE65391", "probes.rds"), art("GSE65391", "metadata.rds"),
                raw("GSE65391", "GPL10558.soft.gz"), file.path(NCBI, "Homo_sapiens.gene_info.gz"), JDM_DE)
absent <- need_files[!file.exists(need_files)]
if (length(absent)) stop("Missing file(s):\n  ", paste(absent, collapse = "\n  "),
                         "\nNotebooks 001 to 003 (public array data) and 05 and 07 (your results) must have run first.")
nih_files <- list.files(DE_DIR, pattern = "^de_.*_vs_HC\\.csv$", full.names = TRUE)
if (!length(nih_files)) stop("No de_<disease>_vs_HC.csv in ", DE_DIR, ": run notebook 05 first.")
read_set <- function(f) { p <- file.path(GENE_SET_DIR, f); if (!file.exists(p)) return(NULL)
  x <- trimws(readLines(p, warn = FALSE)); x[x != "" & !startsWith(x, "#")] }
irg <- read_set("ifn-response-genes-28.txt")
cat("settings OK;", length(nih_files), "RNA-seq disease files found; output folder:", OUT_DIR, "\n")

---
# Part 2 · The array data: bridge the two batches
The arrays were run in two batches. 24 healthy children had the same blood RNA put on two arrays (GEO `set = Technical_Replicate`); for 23 of them one array is in batch 1 and the other in batch 2. Each pair is the **same RNA measured twice**, so any difference inside a pair is batch plus noise. For every probe, the mean of those 23 differences (batch 2 minus batch 1, on the log2 scale) is the batch shift, and it is added to the batch-1 arrays. The technical-replicate arrays are then dropped. This is Anne's notebook 03 (replicate bridging).

**Check printed:** the replicate pairs should differ much less after the shift than before.

In [ ]:
p    <- readRDS(art("GSE65391", "probes.rds"))
meta <- readRDS(art("GSE65391", "metadata.rds"))
E    <- p$E
stopifnot(identical(colnames(E), rownames(meta)) || !anyNA(match(colnames(E), rownames(meta))))
meta <- meta[colnames(E), ]

h <- meta[meta$disease == "Healthy", ]
pairs <- split(rownames(h), paste(h$subject, h$visit))
pairs <- pairs[vapply(pairs, function(pr) setequal(h[pr, "batch"], c("1", "2")), logical(1))]
b1 <- vapply(pairs, function(pr) pr[h[pr, "batch"] == "1"][1], "")
b2 <- vapply(pairs, function(pr) pr[h[pr, "batch"] == "2"][1], "")
D     <- E[, b2] - E[, b1]
shift <- rowMeans(D)
E_c   <- E
in_b1 <- meta$batch == "1"
E_c[, in_b1] <- E[, in_b1] + shift
D_after <- E_c[, b2] - E_c[, b1]
print(c(replicate_pairs = length(pairs), probes = nrow(E), arrays = ncol(E),
        median_abs_pair_difference_before = round(median(abs(D)), 3),
        median_abs_pair_difference_after  = round(median(abs(D_after)), 3)))
if (length(pairs) < 10) stop("Fewer than 10 replicate pairs found: the batch bridge cannot be trusted. Check metadata.rds (batch, set, subject, visit).")

keep_arr <- meta$set != "Technical_Replicate"
E_c <- E_c[, keep_arr]; meta <- meta[keep_arr, ]
print(table(disease = meta$disease))

---
# Part 3 · Probes to genes
Only probes rated Perfect or Good (notebook 003) **and** with an Entrez gene ID are used. Where several probes measure one gene, the probe with the **highest mean expression** is kept (the "MaxMean" rule of `collapseRows()`, Miller JA et al. *BMC Bioinformatics* 2011;12:322; Anne's 03b). Entrez IDs become current NCBI gene symbols; an ID retired since the array was designed keeps the array's own symbol. **expressed** = detected (p < 0.05) in at least 10% of arrays.

In [ ]:
gi  <- read.delim(file.path(NCBI, "Homo_sapiens.gene_info.gz"), quote = "", colClasses = "character")
gpl <- Table(suppressMessages(getGEO(filename = raw("GSE65391", "GPL10558.soft.gz"))))

use <- which(p$quality_ok & !is.na(p$entrez))
mu  <- rowMeans(E_c[use, ])
o   <- use[order(mu, decreasing = TRUE)]
best <- o[!duplicated(p$entrez[o])]                       # per Entrez gene: the best-expressed probe
G <- E_c[best, ]; entrez <- p$entrez[best]
symbol <- gi$Symbol[match(entrez, gi$GeneID)]
old    <- gpl$Symbol[match(entrez, gpl$Entrez_Gene_ID)]
symbol[is.na(symbol)] <- old[is.na(symbol)]
ok <- !is.na(symbol) & !duplicated(symbol)
G <- G[ok, ]; rownames(G) <- symbol[ok]
expressed <- unname(p$detected[best][ok])
print(c(probes_total = nrow(E_c), probes_perfect_good_with_entrez = length(use), genes = nrow(G), genes_expressed = sum(expressed)))

---
# Part 4 · SLE against healthy children (the array study's own comparison)
**Samples** (as Anne's notebook 48 and 06c): the **first visit** of each child with SLE, and **one array per healthy child** (the earliest visit). **Model:** limma, `~ group`, on the expressed genes. A positive log fold change means higher in SLE.

**Sanity check against Anne's published run on the same data:** about 157 children with SLE, 46 healthy children, 13,259 expressed genes, 5,335 genes with adjusted p < 0.05. Yours should be close; a very different number means a step above went differently.

In [ ]:
m <- meta
sle_idx <- which(m$disease == "SLE" & m$first_visit)
hh <- which(m$disease == "Healthy"); hh <- hh[order(m$subject[hh], m$visit[hh])]
hc_idx <- hh[!duplicated(m$subject[hh])]
keep_s <- c(sle_idx, hc_idx)
grp <- factor(ifelse(m$disease[keep_s] == "SLE", "SLE", "healthy"), levels = c("healthy", "SLE"))
X   <- G[expressed, keep_s]
fit <- eBayes(lmFit(X, model.matrix(~ grp)))
sle <- topTable(fit, coef = "grpSLE", number = Inf, sort.by = "P")
sle$gene_name <- rownames(sle)
write.csv(sle, file.path(OUT_DIR, "de_SLE_vs_healthy_GSE65391.csv"), row.names = FALSE)
res4 <- c(genes_tested = nrow(X), SLE_children = sum(grp == "SLE"), healthy_children = sum(grp == "healthy"),
          adj_p_below_0.05 = sum(sle$adj.P.Val < ALPHA),
          top_up = sum(sle$adj.P.Val < ALPHA & sle$logFC >= SLE_LFC), top_down = sum(sle$adj.P.Val < ALPHA & sle$logFC <= -SLE_LFC))
print(res4)

---
# Part 5 · SLE against each disease, gene by gene
For each disease, its `de_<disease>_vs_HC.csv` (notebook 05; JDM from notebook 07) is matched to SLE-vs-healthy **by gene symbol**. The table gives, per disease:
- `rho_all_genes`: Spearman rank correlation of the two fold-change lists (near 1 = same genes move the same way, near 0 = unrelated, negative = opposite);
- `rho_without_ifn`: the same **after removing the 28 interferon response genes** from both sides. If SLE looks like a disease only because both have an interferon signature, this drops;
- `top_in_both` and `top_in_both_same_direction`: genes that pass the cut-off on both sides, and how many go the same way (cut-offs: `SLE_LFC` for SLE, `LFC` for your diseases, `JDM_LFC` for JDM).

`reading`: `holds without interferon genes` = rho stays above 0.2 and at least half its size; `mostly interferon` = it falls away; `no similarity to begin with` = rho was 0.2 or less. Lean on the correlations, not the top-gene counts, because the cut-offs differ between data types.

In [ ]:
others <- list()
for (f in nih_files) {
  dis <- sub("_vs_HC$", "", sub("^de_", "", sub("\\.csv$", "", basename(f))))
  if (dis %in% SKIP_DISEASES) next
  n <- read.csv(f); n <- n[order(-n$AveExpr), ]; n <- n[!duplicated(n$gene_name), ]
  others[[dis]] <- list(d = n, lfc = LFC)
}
jd <- read.csv(JDM_DE); jd <- jd[!duplicated(jd$gene_name), ]
others[["JDM"]] <- list(d = jd, lfc = JDM_LFC)

sle1 <- sle[!duplicated(sle$gene_name), c("gene_name", "logFC", "adj.P.Val")]
one <- function(m) {
  sn <- m$adj.P.Val_sle < ALPHA & abs(m$logFC_sle) >= SLE_LFC
  list(sn = sn, same = sign(m$logFC_sle) == sign(m$logFC_other))
}
rows <- lapply(names(others), function(dis) {
  o <- others[[dis]]
  m <- merge(sle1, o$d[, c("gene_name", "logFC", "adj.P.Val")], by = "gene_name", suffixes = c("_sle", "_other"))
  m$ifn_gene <- if (is.null(irg)) FALSE else m$gene_name %in% irg
  so <- m$adj.P.Val_other < ALPHA & abs(m$logFC_other) >= o$lfc
  a <- one(m); both <- a$sn & so
  m0 <- m[!m$ifn_gene, ]
  rho_all <- cor(m$logFC_sle, m$logFC_other, method = "spearman")
  rho_no  <- cor(m0$logFC_sle, m0$logFC_other, method = "spearman")
  write.csv(m, file.path(OUT_DIR, sprintf("compare_SLE_vs_%s.csv", dis)), row.names = FALSE)
  data.frame(disease = dis, genes_compared = nrow(m), ifn_genes_in_comparison = sum(m$ifn_gene),
             rho_all_genes = round(rho_all, 3), rho_without_ifn = round(rho_no, 3),
             top_sle = sum(a$sn), top_disease = sum(so), top_in_both = sum(both),
             top_in_both_same_direction = sum(both & a$same),
             stringsAsFactors = FALSE)
})
cmp <- do.call(rbind, rows); rownames(cmp) <- NULL
cmp$reading <- ifelse(cmp$rho_all_genes <= 0.2, "no similarity to begin with",
               ifelse(cmp$rho_without_ifn > 0.2 & cmp$rho_without_ifn >= 0.5 * cmp$rho_all_genes,
                      "holds without interferon genes", "mostly interferon"))
print(cmp, right = FALSE)
write.csv(cmp, file.path(OUT_DIR, "comparison_sle_vs_others.csv"), row.names = FALSE)

options(repr.plot.width = 4.5 * length(rows), repr.plot.height = 4.5)
par(mfrow = c(1, length(rows)), mar = c(4, 4, 3, 1))
for (dis in names(others)) {
  mm <- read.csv(file.path(OUT_DIR, sprintf("compare_SLE_vs_%s.csv", dis)))
  plot(mm$logFC_other, mm$logFC_sle, pch = 16, cex = 0.3, col = adjustcolor("grey30", 0.35),
       xlab = sprintf("%s vs its healthy (log2 FC)", dis), ylab = "SLE vs healthy children (log2 FC)",
       main = sprintf("%s: rho = %.2f", dis, cmp$rho_all_genes[cmp$disease == dis]), cex.main = 0.9)
  abline(h = 0, v = 0, lty = 2, col = "grey60")
  if (any(mm$ifn_gene)) points(mm$logFC_other[mm$ifn_gene], mm$logFC_sle[mm$ifn_gene], pch = 16, cex = 0.9, col = "#B2182B")
}

---
# Part 6 · Interferon score of SLE
The score of a sample is the mean over the 28 interferon response genes of its z-score **against the healthy children of the same array study** (value minus their mean, divided by their standard deviation; a floor of 0.1 on the spread). A score near 0 is healthy-like. This is the same definition as in 06 and 07, so the numbers sit on one axis with those notebooks' tables. Note the array does not measure four of the 28 genes (HERC5, HERC6, IFIT5, OAS3 have no Perfect or Good probe; Anne's 48), so the score uses the rest.

SLE scores here are in standard deviations of the **array's** healthy children; compare direction and rank with the other tables, not the exact size.

In [ ]:
if (is.null(irg)) {
  cat("Gene list not found in", GENE_SET_DIR, "- skipped. Fix GENE_SET_DIR in Part 1.\n")
} else {
  g <- intersect(irg, rownames(G))
  cat("interferon genes measured on the array:", length(g), "of", length(irg), "\n")
  hc_cols <- which(grp == "healthy")
  Xg <- G[g, keep_s, drop = FALSE]
  H  <- Xg[, hc_cols, drop = FALSE]
  Z  <- (Xg - rowMeans(H)) / pmax(apply(H, 1, sd), 0.1)
  sc <- colMeans(Z)
  tab <- data.frame(group = c("HC (GSE65391)", "SLE"),
                    samples = c(sum(grp == "healthy"), sum(grp == "SLE")),
                    median_ifn_score = round(c(median(sc[grp == "healthy"]), median(sc[grp == "SLE"])), 2),
                    genes_used = length(g))
  print(tab, row.names = FALSE)
  write.csv(tab, file.path(OUT_DIR, "ifn_score_sle.csv"), row.names = FALSE)
  options(repr.plot.width = 5, repr.plot.height = 4.5)
  boxplot(sc ~ grp, ylab = "interferon score (z against array healthy children)", xlab = "", col = c("grey80", "#7570B3"))
  abline(h = 0, lty = 2, col = "grey50")
}

---
# Part 7 · Log

In [ ]:
log_lines <- c("# SLE comparison log (notebook 08)", paste0("Generated ", format(Sys.time(), "%Y-%m-%d %H:%M"), "\n"),
               "## Settings", paste0("ALPHA=", ALPHA, "; LFC=", LFC, "; JDM_LFC=", JDM_LFC, "; SLE_LFC=", SLE_LFC,
                                     "; skipped: ", paste(SKIP_DISEASES, collapse = ", ")), "",
               "## SLE vs healthy (GSE65391)", "```", capture.output(print(res4)), "```", "",
               "## SLE against each disease", "```", capture.output(print(cmp, right = FALSE)), "```")
writeLines(log_lines, file.path(OUT_DIR, "PROCESSING_LOG_sle.md"))
print(list.files(OUT_DIR))

---
## What to check
1. **Part 2:** `replicate_pairs` should be 23, and the pair difference should be clearly smaller after the shift than before.
2. **Part 4:** close to Anne's numbers (157 SLE, 46 healthy, ~13,259 genes, ~5,335 significant).
3. **Part 5:** a clearly positive `rho_all_genes` with `holds without interferon genes` means SLE shares more with that disease than the interferon response. `mostly interferon` means the resemblance is the interferon signature and little else. Negative or near zero means they differ.
4. **Part 6:** SLE should score above its healthy children.

## Limits
- **Different technologies.** Array and RNA-seq are never put on one scale; only fold changes relative to each study's own healthy controls are compared, by rank and direction.
- **Different blood.** GSE65391 is whole blood from children with SLE on treatment (visit-based); the RGM samples exclude post-treatment samples. Medication can lower interferon signal.
- **Sex and age** are not adjusted for in either comparison.
- **Healthy children** differ in age and recruitment between datasets.

## References
1. Banchereau R et al. Personalized immunomonitoring uncovers molecular networks that stratify lupus patients. *Cell* 2016;165:551-565.
2. Miller JA, Cai C, Langfelder P et al. Strategies for aggregating gene expression data: the collapseRows R function. *BMC Bioinformatics* 2011;12:322.
3. Ritchie ME et al. limma powers differential expression analyses for RNA-sequencing and microarray studies. *Nucleic Acids Res* 2015;43:e47.
4. Kim H et al. *J Interferon Cytokine Res* 2018;38:171-185; de Jesus AA et al. *J Clin Invest* 2020;130:1669-1682 (interferon response genes).